## Generative AI Model Selection & Setup

In [8]:
import os
import sys
import subprocess

# Install required packages in the current notebook environment if missing
def install_if_missing(package):
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_if_missing("openai")
install_if_missing("python-dotenv")

from openai import OpenAI
from dotenv import load_dotenv

# Optional: loads .env only if it exists locally
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. Please set it as an environment variable before running this notebook."
    )

client = OpenAI(api_key=api_key)

In [9]:
# ============================================================
# Section 1: Generative AI Model Selection & Setup
# ============================================================

"""
Model Selected: GPT-4o-mini by OpenAI

Rationale:
- Industry-standard LLM with excellent instruction-following ability
- Fast response times and cost-effective for testing multiple templates
- Strong performance on structured, domain-specific advice generation
- Well-documented API that supports role-based prompting (system/user)
- Widely used in production advice and recommendation systems
"""

import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def call_gpt(prompt: str, system_message: str = None, max_tokens: int = 500) -> str:
    """
    Helper function to call GPT and return the response text.

    Args:
        prompt: The user's message/prompt
        system_message: Optional system-level instruction to set the model's role
        max_tokens: Maximum response length

    Returns:
        str: The model's text response
    """
    messages = []

    if system_message:
        messages.append({"role": "system", "content": system_message})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.7  # slight creativity while staying factual
    )

    return response.choices[0].message.content

# ---- Test Connection ----
test = call_gpt("Confirm you are connected and working in one sentence.")
print("✅ API Connection Successful!")
print(test)

✅ API Connection Successful!
I am connected and ready to assist you!


# Prompt Template Design Documentation

## Template 1: Basic Advice

In [10]:
# ============================================================
# Template 1 (T1): Basic Advice
# ============================================================

"""
Template Name: Basic Advice
Template ID: T1

Intended Use Case:
    Provide a short, plain-language explanation of the ML model's
    prediction. Designed for general users who need simple,
    actionable advice without medical/technical jargon.

Design Rationale:
    This is the baseline template. It asks GPT to respond in plain
    English only, with no structured sections. The goal is to test
    if minimal prompting still produces useful output. This forms the
    comparison baseline for all other templates.

Prompt Structure:
    A patient has the following characteristics:
    {features}

    Our system predicted: {prediction}

    Please give a simple, plain-language explanation of this result
    and basic advice. Keep the response under 120 words.
    Avoid technical jargon.

Assumptions & Limitations:
    - No system role defined — model behaves as general assistant
    - Does not ask for structured output, so format may vary
    - May be too brief for complex cases
    - Best suited as a quick summary for non-expert users
"""

TEMPLATE_1 = """A patient has the following characteristics:
{features}

Our system predicted: {prediction}

Please give a simple, plain-language explanation of this result and basic advice. Keep the response under 120 words. Avoid technical jargon."""

def run_template_1(features: str, prediction: str) -> str:
    prompt = TEMPLATE_1.format(features=features, prediction=prediction)
    return call_gpt(prompt)  # No system message — intentionally minimal

# ---- Example Input/Output ----
example_features = "Gender: Female, Age: 30, Height: 1.60m, Weight: 95kg, BMI: 37.1\nFamily history of overweight: yes\nFrequent high-caloric food (FAVC): yes\nPhysical activity frequency (FAF): 0/3\nMain transportation: Automobile"
example_prediction = "Obesity Type II"

print("=" * 50)
print("TEMPLATE 1 — Basic Advice")
print("=" * 50)
output_t1_example1 = run_template_1(example_features, example_prediction)
print(output_t1_example1)

TEMPLATE 1 — Basic Advice
The prediction of Obesity Type II means that your weight is significantly higher than what is considered healthy for your height. This can increase the risk of health problems. Factors like your family history, eating habits, and lack of physical activity contribute to this condition. 

To improve your health, try to make small changes: 
1. Incorporate more fruits and vegetables into your meals.
2. Reduce high-calorie foods.
3. Aim for at least 30 minutes of physical activity most days, like walking.
4. Consider using public transport or walking short distances instead of driving. 

Consult a healthcare professional for personalized guidance.


## Template 2: Detailed Clinical Explanation

In [11]:
# ============================================================
# Template 2 (T2): Detailed Clinical Explanation
# ============================================================

"""
Template Name: Detailed Clinical Explanation
Template ID: T2

Intended Use Case:
    Provides a structured, thorough explanation for caregivers,
    nursing students, or informed users who want to understand the
    clinical reasoning. Significantly more detailed than T1.

Design Rationale:
    Unlike T1, this template:
    1. Assigns GPT a specific expert role via a system message
    2. Asks for a clearly structured response (4 numbered sections)
    3. Targets an audience with basic health literacy

    The system message primes GPT to behave as a medical advisor,
    improving accuracy and tone consistency across test cases.
    The structured output also makes it easier to evaluate
    completeness in the comparative analysis.

Prompt Structure:
    System: You are a clinical decision-support assistant.
            Provide structured, evidence-based explanations.

    User:
    Patient Profile:
    {features}

    ML Model Prediction: {prediction}

    Provide a structured response with these four sections:
    1. Prediction Explanation — Why did the model predict this?
    2. Key Risk Factors — Which features are most significant?
    3. Recommended Actions — What steps should the patient take?
    4. Warning Signs — What symptoms need immediate attention?

Assumptions & Limitations:
    - System message may cause responses to be overly formal
    - Structured format works best for multi-feature inputs
    - Section 2 (risk factors) relies on model inference, not
      actual feature importance values from the ML model
"""

SYSTEM_T2 = "You are a clinical decision-support assistant. Provide structured, evidence-based explanations to help caregivers and healthcare students understand ML-driven medical predictions."

TEMPLATE_2 = """Patient Profile:
{features}

ML Model Prediction: {prediction}

Provide a structured response with exactly these four sections:
1. Prediction Explanation — Why did the model likely predict this?
2. Key Risk Factors — Which features are most clinically significant?
3. Recommended Actions — What specific steps should be taken?
4. Warning Signs — What symptoms require immediate medical attention?

Keep each section to 2-3 sentences. Target audience: healthcare students and caregivers."""

def run_template_2(features: str, prediction: str) -> str:
    prompt = TEMPLATE_2.format(features=features, prediction=prediction)
    return call_gpt(prompt, system_message=SYSTEM_T2)

# ---- Example Input/Output ----
print("=" * 50)
print("TEMPLATE 2 — Detailed Clinical Explanation")
print("=" * 50)
output_t2_example1 = run_template_2(example_features, example_prediction)
print(output_t2_example1)

TEMPLATE 2 — Detailed Clinical Explanation
### 1. Prediction Explanation
The model likely predicted Obesity Type II due to the patient's significantly high Body Mass Index (BMI) of 37.1, which categorizes her as obese. Additionally, her lack of physical activity (0/3) and consumption of frequent high-caloric foods contribute heavily to her overall energy balance and weight management challenges.

### 2. Key Risk Factors
The most clinically significant features include the patient's elevated BMI, sedentary lifestyle, and dietary habits characterized by frequent consumption of high-caloric foods. Additionally, her family history of overweight suggests a genetic predisposition, which can further exacerbate her risk of obesity and related health issues.

### 3. Recommended Actions
It is crucial to develop a comprehensive weight management plan that includes dietary modification, such as reducing high-caloric food intake, and incorporating regular physical activity tailored to her abilities

## Template 3: Cluster-Aware Personalized Advice

In [12]:
# ============================================================
# Template 3 (T3): Cluster-Aware Personalized Advice
# ============================================================
# This template is different from T1 and T2 because it uses
# clustering results from Phase 2 Part A.
# It tells GPT which lifestyle group the patient belongs to,
# so the advice becomes more personalized.
# ============================================================

"""
Template Name: Cluster-Aware Personalized Advice
Template ID: T3

Intended Use Case:
    Uses the clustering results (K-Means, k=5) to personalize advice.
    GPT receives the patient's cluster info alongside their features,
    so it can tailor recommendations to their lifestyle group.

Design Rationale:
    T1 and T2 only use individual features and the prediction.
    T3 adds cluster information from Phase 2 Part A to give
    group-level context. This tests whether adding cluster info
    improves the personalization of the generated advice.
    This also fulfills the project requirement to integrate
    unsupervised learning with generative AI.

Prompt Structure:
    System: You are a personalized health advisor that uses both
            individual data and population-level lifestyle patterns.
    User: Individual profile + prediction + cluster ID + cluster profile
          → Response in 3 sections: Health Status, Group Insights, Action Plan

Assumptions & Limitations:
    - Depends on cluster quality (silhouette ~0.30)
    - Cluster profiles are manually interpreted from the data
    - If cluster assignment is wrong, advice may be misleading
"""

# --- Cluster profiles from Phase 2 Part A (K-Means, k=5) ---
CLUSTER_PROFILES = {
    0: "Older age group with moderate BMI, moderate physical activity, high caloric food consumption. Most common class: Obesity Type I.",
    1: "Young adults with moderate BMI, moderate activity, high caloric food intake. Most common class: Obesity Type II.",
    2: "Young individuals with low BMI, moderate activity, lower caloric food consumption. Most common class: Insufficient Weight.",
    3: "Highest BMI group, lowest physical activity, very high caloric food intake. Most common class: Obesity Type III.",
    4: "Moderate BMI, highest technology use time, moderate physical activity. Most common class: Obesity Type I."
}

# --- System message: tells GPT its role ---
SYSTEM_T3 = "You are a personalized health advisor that uses both individual patient data and population-level lifestyle patterns identified through clustering analysis. Tailor your advice based on the patient's lifestyle group."

# --- The prompt template ---
TEMPLATE_3 = """Individual Profile:
{features}

ML Prediction: {prediction}

Lifestyle Cluster: Cluster {cluster_id}
Cluster Profile: {cluster_profile}

Based on both the individual profile and the lifestyle cluster they belong to, provide personalized advice with exactly these three sections:

1. Your Health Status — Summarize the patient's current health classification and what it means.
2. Your Lifestyle Group Insights — Explain what their cluster reveals about shared habits and risks among similar individuals.
3. Personalized Action Plan — Give 3-4 specific, actionable recommendations tailored to both their individual data AND their cluster's tendencies.

Keep the total response under 250 words. Use encouraging, supportive language."""

# --- Function to run Template 3 ---
def run_template_3(features, prediction, cluster_id):
    cluster_profile = CLUSTER_PROFILES.get(cluster_id, "Unknown cluster profile")
    prompt = TEMPLATE_3.format(
        features=features,
        prediction=prediction,
        cluster_id=cluster_id,
        cluster_profile=cluster_profile
    )
    return call_gpt(prompt, system_message=SYSTEM_T3)

# ---- Example Input/Output ----
example_features = "Gender: Female, Age: 30, Height: 1.60m, Weight: 95kg, BMI: 37.1\nFamily history of overweight: yes\nFrequent high-caloric food (FAVC): yes\nPhysical activity frequency (FAF): 0/3\nMain transportation: Automobile"
example_prediction = "Obesity Type II"
example_cluster = 3

print("=" * 50)
print("TEMPLATE 3 — Cluster-Aware Personalized Advice")
print("=" * 50)
output_t3_example = run_template_3(example_features, example_prediction, example_cluster)
print(output_t3_example)

TEMPLATE 3 — Cluster-Aware Personalized Advice
### Your Health Status  
You are classified as having Obesity Type II, which means your weight is significantly above the healthy range for your height, increasing your risk for various health issues such as diabetes, heart disease, and joint problems. Recognizing this is the first step towards positive change, and there’s plenty you can do to improve your health.

### Your Lifestyle Group Insights  
As part of Cluster 3, you share common habits with individuals who have the highest BMI, lowest physical activity levels, and very high caloric food intake. This group often struggles with making healthy choices, which can lead to increased health risks. However, understanding these patterns provides a unique opportunity to shift towards healthier habits together.

### Personalized Action Plan  
1. **Gradual Dietary Changes**: Start by incorporating one healthy meal or snack per day, focusing on whole foods like fruits, vegetables, and whole g

## Template 4: Motivational Behavior-Change Coach

In [13]:
# ============================================================
# Template 4 (T4): Motivational Behavior-Change Coach
# ============================================================
# This template focuses on HOW the advice is delivered.
# Instead of clinical or technical language, it uses a
# coaching tone with small, achievable steps.
# The goal is to test whether a motivational style produces
# more engaging advice compared to T1 (brief), T2 (clinical),
# and T3 (cluster-based).
# ============================================================

"""
Template Name: Motivational Behavior-Change Coach
Template ID: T4

Intended Use Case:
    For end-users (patients) who need motivational, non-intimidating
    guidance focused on gradual behavior change.

Design Rationale:
    Research shows patients follow advice more when it is:
    - Empathetic and non-judgmental
    - Broken into small, achievable steps
    - Focused on positive framing ("you can" vs "you must")
    This tests whether coaching-style advice is more engaging
    than the clinical style of T2 or the brevity of T1.

Prompt Structure:
    System: You are a supportive health coach.
    User: Patient features + prediction
          → Response in 3 sections: Understanding, Small Steps, Motivation

Assumptions & Limitations:
    - May sound overly optimistic for severe obesity cases
    - Coaching tone might not suit users who want direct facts
    - Does not replace professional medical advice
"""

# --- System message: coaching persona ---
SYSTEM_T4 = "You are a warm, supportive health and wellness coach specializing in weight management and lifestyle improvement. You believe in small, sustainable changes rather than dramatic overhauls. Never use fear-based language or medical jargon. Always be encouraging."

# --- The prompt template ---
TEMPLATE_4 = """About this person:
{features}

Health assessment result: {prediction}

As their personal health coach, write a warm and motivational response with these three sections:

1. Understanding Your Result — Explain what the health assessment means in simple, non-scary terms. Validate their feelings.
2. Small Steps This Week — Suggest 3 specific, easy-to-start actions they can begin THIS WEEK (not long-term goals). Make them feel achievable.
3. Your Motivation Boost — End with an encouraging message that acknowledges their effort in seeking help and inspires them to take the first step.

Keep the total response under 200 words. Use "you" language and a friendly, coaching tone. Avoid words like "obese", "dangerous", or "risk"."""

# --- Function to run Template 4 ---
def run_template_4(features, prediction):
    prompt = TEMPLATE_4.format(features=features, prediction=prediction)
    return call_gpt(prompt, system_message=SYSTEM_T4)

# ---- Example Input/Output ----
example_features = "Gender: Female, Age: 30, Height: 1.60m, Weight: 95kg, BMI: 37.1\nFamily history of overweight: yes\nFrequent high-caloric food (FAVC): yes\nPhysical activity frequency (FAF): 0/3\nMain transportation: Automobile"
example_prediction = "Obesity Type II"
example_cluster = 3

print("=" * 50)
print("TEMPLATE 3 — Cluster-Aware Personalized Advice")
print("=" * 50)
output_t3_example = run_template_3(example_features, example_prediction, example_cluster)
print(output_t3_example)

TEMPLATE 3 — Cluster-Aware Personalized Advice
1. **Your Health Status**  
You have a BMI of 37.1, which classifies you as Obesity Type II. This means you are at a higher risk for health complications such as type 2 diabetes, heart disease, and joint problems. Recognizing this can empower you to make positive changes toward a healthier lifestyle.

2. **Your Lifestyle Group Insights**  
As part of Cluster 3, you share habits with individuals who also have the highest BMI, the lowest physical activity levels, and a tendency for high-caloric food intake. This cluster often faces similar challenges, including emotional eating and limited mobility, which can further complicate weight management.

3. **Personalized Action Plan**  
- **Start Small with Physical Activity:** Aim for just 10-15 minutes of light activity daily, such as walking or dancing at home. Gradually increase the duration as it becomes more comfortable.  
- **Mindful Eating:** Keep a food diary to track your meals and snack

# Testing Framework & Output Comparison

## Testing Framework

In [15]:
# ============================================================
# Testing Framework
# ============================================================
# Here we pick 3 different patients from our real dataset.
# We chose diverse cases on purpose:
#   - One severe case (Obesity Type III)
#   - One healthy case (Normal Weight)
#   - One in-between case (Overweight Level I)
# This way we can see how each template handles different situations.
# ============================================================

import pandas as pd

# Load both raw data (readable values) and clustered data (has cluster labels)
raw = pd.read_csv("Dataset/raw_kaggle_data.csv")
clustered = pd.read_csv("Dataset/clustered_data.csv")

# --- Helper function ---
# This takes a row from the dataset and turns it into a readable
# text that we can send to GPT (instead of raw numbers)
def format_features(row):
    bmi = row["Weight"] / (row["Height"] ** 2)
    return (
        f"Gender: {row['Gender']}, Age: {int(row['Age'])}, "
        f"Height: {row['Height']}m, Weight: {row['Weight']}kg, BMI: {bmi:.1f}\n"
        f"Family history of overweight: {row['family_history_with_overweight']}\n"
        f"Frequent high-caloric food (FAVC): {row['FAVC']}\n"
        f"Vegetable consumption (FCVC): {row['FCVC']}/3, "
        f"Meals per day (NCP): {row['NCP']}\n"
        f"Eating between meals (CAEC): {row['CAEC']}\n"
        f"Smoker: {row['SMOKE']}, Daily water (CH2O): {row['CH2O']}L\n"
        f"Calorie monitoring (SCC): {row['SCC']}\n"
        f"Physical activity frequency (FAF): {row['FAF']}/3\n"
        f"Technology use time (TUE): {row['TUE']}hrs\n"
        f"Alcohol consumption (CALC): {row['CALC']}\n"
        f"Main transportation: {row['MTRANS']}"
    )

# ============================================================
# Pick our 3 test cases
# ============================================================

# Test Case 1: Someone with severe obesity
tc1_raw = raw[raw["NObeyesdad"] == "Obesity_Type_III"].iloc[0]
tc1_cluster = int(clustered[clustered["NObeyesdad_encoded"] == 4].iloc[0]["cluster"])

# Test Case 2: Someone with normal weight
tc2_raw = raw[raw["NObeyesdad"] == "Normal_Weight"].iloc[5]
tc2_cluster = int(clustered[clustered["NObeyesdad_encoded"] == 1].iloc[5]["cluster"])

# Test Case 3: Someone slightly overweight
tc3_raw = raw[raw["NObeyesdad"] == "Overweight_Level_I"].iloc[0]
tc3_cluster = int(clustered[clustered["NObeyesdad_encoded"] == 5].iloc[0]["cluster"])

# Put them in a list so we can loop through them easily
test_cases = [
    {
        "id": "TC1",
        "description": "Severe Obesity (Obesity Type III)",
        "features": format_features(tc1_raw),
        "prediction": "Obesity Type III",
        "cluster": tc1_cluster
    },
    {
        "id": "TC2",
        "description": "Normal Weight",
        "features": format_features(tc2_raw),
        "prediction": "Normal Weight",
        "cluster": tc2_cluster
    },
    {
        "id": "TC3",
        "description": "Overweight Level I",
        "features": format_features(tc3_raw),
        "prediction": "Overweight Level I",
        "cluster": tc3_cluster
    }
]

# Let's see what we picked
for tc in test_cases:
    print(f"\n{'='*50}")
    print(f"Test Case: {tc['id']} — {tc['description']}")
    print(f"Cluster: {tc['cluster']}")
    print(f"{'='*50}")
    print(tc["features"])


Test Case: TC1 — Severe Obesity (Obesity Type III)
Cluster: 3
Gender: Female, Age: 26, Height: 1.56m, Weight: 102.0kg, BMI: 41.9
Family history of overweight: yes
Frequent high-caloric food (FAVC): yes
Vegetable consumption (FCVC): 3.0/3, Meals per day (NCP): 3.0
Eating between meals (CAEC): Sometimes
Smoker: yes, Daily water (CH2O): 1.0L
Calorie monitoring (SCC): no
Physical activity frequency (FAF): 0.0/3
Technology use time (TUE): 1.0hrs
Alcohol consumption (CALC): Sometimes
Main transportation: Public_Transportation

Test Case: TC2 — Normal Weight
Cluster: 1
Gender: Male, Age: 22, Height: 1.64m, Weight: 53.0kg, BMI: 19.7
Family history of overweight: no
Frequent high-caloric food (FAVC): no
Vegetable consumption (FCVC): 2.0/3, Meals per day (NCP): 3.0
Eating between meals (CAEC): Sometimes
Smoker: no, Daily water (CH2O): 2.0L
Calorie monitoring (SCC): no
Physical activity frequency (FAF): 3.0/3
Technology use time (TUE): 0.0hrs
Alcohol consumption (CALC): Sometimes
Main transporta

## Running All Templates on Test Cases

In [16]:
# ============================================================
# Running all 4 templates on all 3 test cases
# ============================================================
# This gives us 4 templates × 3 cases = 12 outputs total.
# We run them on the SAME cases so we can compare fairly.
# We also add a small delay between calls so we don't hit
# the API rate limit.
# ============================================================

import time

# We'll store all results here so we can save them later
all_results = {}

for tc in test_cases:
    tc_id = tc["id"]
    all_results[tc_id] = {}

    print(f"\n{'#'*60}")
    print(f"# Running {tc_id}: {tc['description']}")
    print(f"{'#'*60}")

    # --- T1: Basic Advice (from Member 1) ---
    print(f"\n--- T1: Basic Advice ---")
    result_t1 = run_template_1(tc["features"], tc["prediction"])
    all_results[tc_id]["T1"] = result_t1
    print(result_t1)
    time.sleep(1)  # wait 1 second before next call

    # --- T2: Detailed Clinical Explanation (from Member 1) ---
    print(f"\n--- T2: Detailed Clinical Explanation ---")
    result_t2 = run_template_2(tc["features"], tc["prediction"])
    all_results[tc_id]["T2"] = result_t2
    print(result_t2)
    time.sleep(1)

    # --- T3: Cluster-Aware Personalized Advice (ours) ---
    # Notice this one takes cluster_id as extra input
    print(f"\n--- T3: Cluster-Aware Personalized Advice ---")
    result_t3 = run_template_3(tc["features"], tc["prediction"], tc["cluster"])
    all_results[tc_id]["T3"] = result_t3
    print(result_t3)
    time.sleep(1)

    # --- T4: Motivational Behavior-Change Coach (ours) ---
    print(f"\n--- T4: Motivational Behavior-Change Coach ---")
    result_t4 = run_template_4(tc["features"], tc["prediction"])
    all_results[tc_id]["T4"] = result_t4
    print(result_t4)
    time.sleep(1)

print("\n✅ Done! All 12 test runs completed.")


############################################################
# Running TC1: Severe Obesity (Obesity Type III)
############################################################

--- T1: Basic Advice ---
The prediction of Obesity Type III means that you have a high level of obesity, which can lead to health issues. Your weight is significantly higher than what is considered healthy for your height. To improve your health, try to eat more healthy foods, like fruits and vegetables, and reduce high-calorie snacks. It's also important to be more active, even small activities like walking can help. Drinking more water and cutting back on smoking and alcohol would be beneficial too. Consider talking to a healthcare professional for personalized advice and support.

--- T2: Detailed Clinical Explanation ---
### 1. Prediction Explanation
The model likely predicted Obesity Type III due to the patient's high BMI of 41.9, indicating severe obesity. Additionally, the combination of high caloric intake, 

## Saving Outputs

In [18]:
# ============================================================
# Save all outputs to files
# ============================================================
# We save each output as a separate .txt file so it's easy
# to review and compare later.
# We also save everything together in one .json file
# ============================================================

import json
import os

# Make sure the folder exists
os.makedirs("Generative_AI/example_outputs", exist_ok=True)

# Template names for labeling the files
template_names = {
    "T1": "Basic Advice",
    "T2": "Detailed Clinical Explanation",
    "T3": "Cluster-Aware Personalized Advice",
    "T4": "Motivational Behavior-Change Coach"
}

# Save each output as a separate .txt file
for tc in test_cases:
    tc_id = tc["id"]
    for t_id in ["T1", "T2", "T3", "T4"]:
        filename = f"Generative_AI/example_outputs/{t_id}_{tc_id.lower()}.txt"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"Template: {t_id} — {template_names[t_id]}\n")
            f.write(f"Test Case: {tc_id} — {tc['description']}\n")
            f.write(f"Input Features:\n{tc['features']}\n")
            f.write(f"Prediction: {tc['prediction']}\n")
            # Only T3 has cluster info
            if t_id == "T3":
                f.write(f"Cluster: {tc['cluster']}\n")
            f.write(f"\n--- OUTPUT ---\n{all_results[tc_id][t_id]}")
        print(f"✅ Saved {filename}")

# Save everything in one JSON file (easier for analysis)
with open("Generative_AI/example_outputs/all_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print("\n✅ All outputs saved!")

✅ Saved Generative_AI/example_outputs/T1_tc1.txt
✅ Saved Generative_AI/example_outputs/T2_tc1.txt
✅ Saved Generative_AI/example_outputs/T3_tc1.txt
✅ Saved Generative_AI/example_outputs/T4_tc1.txt
✅ Saved Generative_AI/example_outputs/T1_tc2.txt
✅ Saved Generative_AI/example_outputs/T2_tc2.txt
✅ Saved Generative_AI/example_outputs/T3_tc2.txt
✅ Saved Generative_AI/example_outputs/T4_tc2.txt
✅ Saved Generative_AI/example_outputs/T1_tc3.txt
✅ Saved Generative_AI/example_outputs/T2_tc3.txt
✅ Saved Generative_AI/example_outputs/T3_tc3.txt
✅ Saved Generative_AI/example_outputs/T4_tc3.txt

✅ All outputs saved!


In [19]:
# ============================================================
# Save all 4 prompt templates to JSON

import json
import os

os.makedirs("Generative_AI/prompts", exist_ok=True)

templates = {
    "T1": {
        "name": "Basic Advice",
        "model": "gpt-4o-mini",
        "system_message": None,
        "template": TEMPLATE_1
    },
    "T2": {
        "name": "Detailed Clinical Explanation",
        "model": "gpt-4o-mini",
        "system_message": SYSTEM_T2,
        "template": TEMPLATE_2
    },
    "T3": {
        "name": "Cluster-Aware Personalized Advice",
        "model": "gpt-4o-mini",
        "system_message": SYSTEM_T3,
        "template": TEMPLATE_3,
        "cluster_profiles": CLUSTER_PROFILES
    },
    "T4": {
        "name": "Motivational Behavior-Change Coach",
        "model": "gpt-4o-mini",
        "system_message": SYSTEM_T4,
        "template": TEMPLATE_4
    }
}

with open("Generative_AI/prompts/templates.json", "w", encoding="utf-8") as f:
    json.dump(templates, f, indent=2, default=str)

print("✅ All 4 templates saved to Generative_AI/prompts/templates.json")

✅ All 4 templates saved to Generative_AI/prompts/templates.json


## Output Comparison

# Analysis: Qualitative & Quantitative Results

# Best Prompt Selection & Justification

# Integration Plan for Final System

# Ethical Considerations & Limitations